![Ironhack logo](https://user-images.githubusercontent.com/23629340/40541063-a07a0a8a-601a-11e8-91b5-2f13e4e6b441.png)

# Multimodal AI
*Data Science & Machine Learning - Week 8, Day 1*

## How to use this notebook

So far, every model in this module has read text and written text. Real problems often come as **photos, scans and voice messages**: a customer sends a picture of a damaged card, an insurance claim comes with photos, a support line receives voice notes. **Multimodal** models read or produce more than one kind of data.

You will:

1. Send **images** to an LLM and ask questions about them, and see what an image costs.
2. Compare images and text **locally** with CLIP, a model that puts both in the same embedding space.
3. Turn text into **speech**, and speech back into **text**.
4. Chain the three into a tiny **voice assistant**.
5. Use these models responsibly: consent, privacy and synthetic media.

The images are in the `img` folder next to this notebook. The **Appendix** has extra material that is not part of the lesson.

## Table of contents

- [Setup](#Setup)
- [1. Images in, text out](#1.-Images-in,-text-out)
- [2. Images and text in one space: CLIP](#2.-Images-and-text-in-one-space:-CLIP)
- [3. Text to speech](#3.-Text-to-speech)
- [4. Speech to text](#4.-Speech-to-text)
- [5. A voice assistant in three calls](#5.-A-voice-assistant-in-three-calls)
- [6. Using multimodal AI responsibly](#6.-Using-multimodal-AI-responsibly)
- [Summary](#Summary)
- [Appendix](#Appendix)

## Setup

You need `pip install openai python-dotenv pillow transformers torch`, and your OpenAI key in the `.env` file, as last week. The CLIP model (about 600 MB) downloads the first time you run section 2. Running the whole notebook costs about two US cents.

In [1]:
import os
import io
import base64
import logging
import warnings

os.environ["HF_HUB_VERBOSITY"] = "error"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.getLogger("torch.distributed.elastic.multiprocessing.redirects").setLevel(logging.ERROR)

import pandas as pd
import torch
from PIL import Image
from IPython.display import Audio, display
from dotenv import load_dotenv
from openai import OpenAI
from transformers import CLIPModel, CLIPProcessor
from transformers.utils import logging as transformers_logging

transformers_logging.set_verbosity_error()
transformers_logging.disable_progress_bar()
warnings.filterwarnings("ignore", message=".*torch.jit.script.*")

load_dotenv()
client = OpenAI()
MODEL = "gpt-5.4-nano"

IMAGES = {name: Image.open(f"img/{name}.jpg").convert("RGB") for name in ["cat", "dog", "dog2", "server"]}
for name, image in IMAGES.items():
    print(f"{name:7s} {image.size[0]} x {image.size[1]} pixels")

cat     1200 x 1602 pixels
dog     2048 x 1365 pixels
dog2    800 x 534 pixels
server  1600 x 900 pixels


## 1. Images in, text out

The GPT-5 models read images as well as text. In the `content` of a user message, you send a **list** of parts instead of a string: a text part with the question, and an image part. A local image is sent as a **data URL**: the image file encoded as text with base64.

Images cost tokens, and big images cost more, so we first shrink each image to at most 768 pixels on its longest side. That is plenty for describing a photo.

In [2]:
def image_part(image, max_side=768):
    image = image.copy()
    image.thumbnail((max_side, max_side))                 # keeps the proportions
    buffer = io.BytesIO()
    image.save(buffer, format="JPEG", quality=85)
    encoded = base64.b64encode(buffer.getvalue()).decode()
    return {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{encoded}"}}

def ask_about_image(image, question, **options):
    response = client.chat.completions.create(model=MODEL, messages=[{"role": "user", "content": [
        {"type": "text", "text": question}, image_part(image, **options)]}])
    return response.choices[0].message.content, response.usage.prompt_tokens

answer, tokens = ask_about_image(IMAGES["server"], "Describe this image in one sentence, then list three objects you can see.")
print(answer, f"\n\n({tokens:,} input tokens)")

The image shows a row of rack-mounted computer servers with glowing indicator lights in a data center.  
Three objects visible are: **server racks**, **cables**, and **LED indicator lights**. 

(424 input tokens)


The answer is a sentence and a list, exactly as if you had described the photo in words yourself. Look at the input tokens: the image costs far more than the question. The model cuts every image into small square patches, and each patch costs tokens, so **the number of pixels sets the price**. Compare the same photo at three sizes:

In [3]:
for max_side in [512, 768, 2048]:
    answer, tokens = ask_about_image(IMAGES["dog"], "What is happening in this photo? One sentence.", max_side=max_side)
    print(f"{max_side:5d} px ({tokens:,} input tokens): {answer}")

  512 px (227 input tokens): A woman is kneeling on the beach and greeting or comforting a sea lion while it sits in the surf.


  768 px (476 input tokens): A woman is kneeling on the beach and playing with a dog as waves roll in behind them.


 2048 px (2,896 input tokens): A woman kneels on the beach smiling as she plays and communicates with her dog while waves roll in behind them.


The full-size photo costs about six times as much as the 768-pixel one, and the answer is no better. Too small is a false economy, though: in the reference run, the 512-pixel version made the model see a **sea lion** instead of a dog. Around 768 pixels is a good default for photos. For small details (text on a document, a serial number, a scratch on a card), send a bigger image, or crop the part that matters. (Older models such as GPT-4o also accept a `"detail": "low"` option in the image part; with the GPT-5.4 models, the size of the image is what counts.)

Everything you learned on Thursday works with images too: a system prompt, structured outputs, function calling. Here a Pydantic class turns photos into rows of a table:

In [4]:
from pydantic import BaseModel
from typing import Literal

class Photo(BaseModel):
    main_subject: Literal["animal", "person", "building", "object", "other"]
    animal: str
    setting: Literal["indoors", "outdoors"]
    people_visible: int

rows = []
for name, image in IMAGES.items():
    response = client.chat.completions.parse(model=MODEL, response_format=Photo, messages=[{"role": "user", "content": [
        {"type": "text", "text": "Describe this photo. Use 'none' for the animal if there is no animal."},
        image_part(image, max_side=512)]}])
    rows.append({"image": name, **response.choices[0].message.parsed.model_dump()})
pd.DataFrame(rows)

,image,main_subject,animal,setting,people_visible
0,cat,animal,tabby cat,outdoors,0
1,dog,person,dog,outdoors,2
2,dog2,animal,dog,outdoors,0
3,server,building,none,indoors,0


This is how photos become data: for example, sorting the photos attached to insurance claims, or extracting the fields of a scanned form. As always, check a sample by hand before you trust the table: these images were sent at 512 pixels, and in the reference run the model counted **two** people on the beach, where there is one.

## 2. Images and text in one space: CLIP

On Day 2 of last week, sentence embeddings put texts with similar meanings close together. **CLIP** (from OpenAI, 2021) does the same for images **and** texts at once: it was trained on 400 million images with their captions, so that a photo and a sentence describing it end up close together. It runs on your computer, for free.

[visual: two encoders, one for images and one for texts, pointing into the same space; the photo of a cat lands next to "a photo of a cat", far from "a server room"]

That makes **zero-shot image classification** possible: compare an image with a few sentences, one per class, and pick the closest. No training, and the classes are whatever you write.

In [5]:
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

labels = ["a photo of a cat", "a photo of a dog", "a photo of a person", "a photo of a data center", "a photo of a car"]
inputs = processor(text=labels, images=list(IMAGES.values()), return_tensors="pt", padding=True)
with torch.no_grad():
    probabilities = clip(**inputs).logits_per_image.softmax(dim=1)

table = pd.DataFrame(probabilities.numpy(), index=list(IMAGES), columns=[label.replace("a photo of a ", "") for label in labels])
table.round(3)

,cat,dog,person,data center,car
cat,0.994,0.002,0.003,0.000,0.000
dog,0.013,0.942,0.043,0.001,0.001
dog2,0.002,0.990,0.008,0.000,0.000
server,0.000,0.000,0.000,1.000,0.000


Each row is an image, each column a label, and each row adds up to 1. The cat, the dogs and the data center get their label with high confidence. Look at `dog`: the photo shows a dog **and** a woman, yet *person* gets only a few percent, because a softmax assumes that exactly one label is right and gives almost everything to the best one. For photos with several subjects, compare the raw similarities instead, or ask an LLM as in section 1, which noticed both.

The same space works the other way round, for **search**: embed every image once, then find the image closest to a text query. This is the idea behind searching your phone's photos for "beach".

In [6]:
with torch.no_grad():
    image_vectors = clip.get_image_features(**processor(images=list(IMAGES.values()), return_tensors="pt")).pooler_output
    image_vectors = image_vectors / image_vectors.norm(dim=-1, keepdim=True)

def search_images(query):
    with torch.no_grad():
        text_vector = clip.get_text_features(**processor(text=[query], return_tensors="pt", padding=True)).pooler_output
        text_vector = text_vector / text_vector.norm(dim=-1, keepdim=True)
    scores = (image_vectors @ text_vector.T).squeeze(1)
    return pd.Series(scores.numpy(), index=list(IMAGES)).sort_values(ascending=False)

for query in ["a pet at the beach at sunset", "computers in a cold room", "an animal sitting on a wall"]:
    ranking = search_images(query)
    print(f"{query:32s} -> {ranking.index[0]:7s} (cosine similarity {ranking.iloc[0]:.3f})")

a pet at the beach at sunset     -> dog     (cosine similarity 0.290)
computers in a cold room         -> server  (cosine similarity 0.259)
an animal sitting on a wall      -> cat     (cosine similarity 0.269)


Every query found the right photo, although no word of the query appears in any file name or caption. CLIP similarities are smaller than the sentence-embedding similarities of Week 7 (around 0.3 here), because images and texts never match exactly; what matters is the ranking.

## 3. Text to speech

**Text to speech (TTS)** turns text into an audio file. OpenAI's `gpt-4o-mini-tts` model takes the text, a **voice** and optional **instructions** about how to speak: the tone, the speed, the emotion.

In [7]:
question_text = "Hello! My card was blocked after I typed the wrong PIN three times. How do I unblock it?"

with client.audio.speech.with_streaming_response.create(
    model="gpt-4o-mini-tts",
    voice="coral",
    input=question_text,
    instructions="Speak like a slightly worried customer calling their bank.",
) as response:
    response.stream_to_file("customer_question.mp3")

print(f"customer_question.mp3: {os.path.getsize('customer_question.mp3') / 1024:.0f} KB")
display(Audio("customer_question.mp3"))

customer_question.mp3: 122 KB


Press play. The voice sounds natural, and the instructions changed *how* it speaks, not *what* it says. Try other voices (`alloy`, `ash`, `nova`, `sage`...) and other instructions, such as *speak slowly and calmly, like a support agent*.

A voice this natural raises an obvious problem: anyone can make a computer say anything in a convincing voice. Keep that in mind for section 6.

## 4. Speech to text

**Speech to text (STT)**, also called transcription, goes the other way. We transcribe the file we just generated, so we know exactly what it should say:

In [8]:
with open("customer_question.mp3", "rb") as audio_file:
    transcript = client.audio.transcriptions.create(model="gpt-4o-mini-transcribe", file=audio_file)

print("Original:  ", question_text)
print("Transcript:", transcript.text)
print("Audio tokens:", transcript.usage.input_tokens)

Original:   Hello! My card was blocked after I typed the wrong PIN three times. How do I unblock it?
Transcript: Hello, my card was blocked after I typed the wrong PIN three times. How do I unblock it?
Audio tokens: 78


Compare it word by word with the original. In some reference runs it was perfect apart from the punctuation; in another, *PIN* came back as *PINs*. Transcription errors are usually this small, and they matter most exactly where the details matter: names, numbers, codes and account details. Real recordings, with background noise, accents and people talking over each other, make more of them.

Transcription is also how you get **text out of meetings, calls and videos** for everything you did in Week 7: classify the call, retrieve the right policy, summarise it.

## 5. A voice assistant in three calls

Put the pieces together: speech to text, an LLM with the bank's policies, and text to speech. This is the skeleton of every voice assistant.

[visual: microphone → speech to text → text question → LLM (with policies) → text answer → text to speech → speaker]

In [9]:
POLICIES = """- After three wrong PIN attempts the card is blocked. Unblock it in the app under Card > PIN > Unblock, or at any of the bank's ATMs.
- A lost or stolen card must be frozen immediately in the app under Card > Freeze.
- New cards arrive within 7 working days."""

def voice_assistant(audio_path, answer_path="assistant_answer.mp3"):
    with open(audio_path, "rb") as audio_file:
        question = client.audio.transcriptions.create(model="gpt-4o-mini-transcribe", file=audio_file).text
    answer = client.chat.completions.create(model=MODEL, messages=[
        {"role": "system", "content": "You are the bank's phone assistant. Answer in two short sentences, only from these policies:\n" + POLICIES},
        {"role": "user", "content": question}]).choices[0].message.content
    with client.audio.speech.with_streaming_response.create(model="gpt-4o-mini-tts", voice="sage", input=answer,
            instructions="Speak calmly and clearly, like a friendly support agent.") as response:
        response.stream_to_file(answer_path)
    return question, answer

question, answer = voice_assistant("customer_question.mp3")
print("Heard: ", question)
print("Answer:", answer)
display(Audio("assistant_answer.mp3"))

Heard:  Hello, my card was blocked after I typed the wrong PIN three times. How do I unblock it?
Answer: After three wrong PIN attempts your card is blocked. You can unblock it in the app under **Card > PIN > Unblock**, or at any of the bank’s ATMs.


Three API calls, and the customer heard an answer based on the bank's policies. Real voice assistants also need to handle interruptions and silence, and to be fast: each call adds latency, which is why OpenAI also offers **realtime** models that listen and speak in one connection. The structure, though, is exactly this one.

## 6. Using multimodal AI responsibly

### Consent and synthetic media

A natural synthetic voice, or a generated image, can be used to deceive. Under the EU AI Act you studied on Friday (Article 50), people must be told when they are talking to an AI system, and synthetic audio, images and video must be marked as artificially generated. In practice: tell callers they are talking to an assistant, never imitate a real person's voice without their consent, and label generated media.

### Photos and recordings are personal data

A photo of a person, a voice recording or a transcript of a call is **personal data** under the GDPR, and faces and voices can be **biometric data**, which has extra protection. Sending them to an API means sending them to a provider: you need a legal basis and your organisation's approval, and you should send only what is needed (crop or blur what is not, and do not keep recordings longer than necessary). Photos can also contain hidden **metadata**, such as the GPS location where they were taken.

### Errors in the details

Image descriptions and transcripts are fluent and usually right, and they fail on exactly the details that matter: a digit in a number, a name, a small object. Check a sample by hand, and never let a transcript or a photo description trigger an action on its own.

### Cost

Images and audio cost more than text: an image is hundreds of input tokens, and audio is billed by length. Shrink images, crop what matters, and process audio only once.

## Summary

| Task | Model | Code |
|---|---|---|
| Ask about an image | `gpt-5.4-nano` | content = `[{"type": "text", ...}, {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64,..."}}]` |
| Make images cheaper | | resize first: the price grows with the number of pixels |
| Images to table rows | `gpt-5.4-nano` | `client.chat.completions.parse(..., response_format=MyModel)` |
| Zero-shot image classification | CLIP, local | `processor(text=labels, images=images)`, `logits_per_image.softmax(dim=1)` |
| Text-to-image search | CLIP, local | normalised `get_image_features(...).pooler_output` and `get_text_features(...).pooler_output`, then a dot product |
| Text to speech | `gpt-4o-mini-tts` | `client.audio.speech.with_streaming_response.create(model=..., voice=..., input=..., instructions=...)` |
| Speech to text | `gpt-4o-mini-transcribe` | `client.audio.transcriptions.create(model=..., file=...)` |

The rules that matter most:

+ **Shrink images** before you send them: the price grows with the number of pixels.
+ **Check the details**: numbers, names and codes are where transcripts and descriptions fail.
+ **Photos and recordings are personal data**, and faces and voices can be biometric data.
+ **Disclose AI voices and label synthetic media**; never imitate a real voice without consent.

In the lab, the stretch task gives your research agent a voice: a spoken question in, a spoken answer out.

## Appendix

### A.1 Transcription on your own computer

OpenAI's **Whisper** models are open, so you can transcribe locally, for free and without sending the audio anywhere. The smallest one is enough for clear speech (reading MP3 files also needs `ffmpeg` installed on your computer):

```python
from transformers import pipeline

transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-tiny")
transcriber("customer_question.mp3")["text"]
```

Bigger Whisper models are more accurate and slower; on a laptop, `whisper-base` or `whisper-small` is a good compromise.

### A.2 Generating images

The same client generates images with the `gpt-image` models. Each image costs a few cents, more than everything else in this notebook together, so the code is shown but not run:

```python
result = client.images.generate(model="gpt-image-1-mini", prompt="A friendly robot helping a customer at a bank counter, flat illustration", size="1024x1024", quality="low")
with open("robot.png", "wb") as file:
    file.write(base64.b64decode(result.data[0].b64_json))
```

Generated images must be labelled as artificially generated when you publish them (section 6).